# 🎙️ Train Custom ALFRED Wake-Word Model (.onnx)
### Using Your Voice & Your Friend's Voice (No Piper / No Phonemize Issues!)

This notebook trains a custom **OpenWakeWord neural network** directly on real voice recordings from you and your friend, plus background noise and chatter to prevent false activations.

When finished, it outputs a ready-to-use **`alfred.onnx`** model file that you can download and drop straight into ALFRED's `models/` folder.

## Step 1: Install Dependencies & Download Embedding Models
We install `openwakeword`, `onnx`, `onnxruntime`, and `audiomentations`. We do **not** install `piper-phonemize`, avoiding Colab Python 3.11+ build errors completely.

In [ ]:
!pip install -q openwakeword onnx onnxruntime torch torchaudio audiomentations scipy tqdm

import os
import glob
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import scipy.io.wavfile as wavfile
from tqdm.auto import tqdm

# Pre-download Google speech feature and embedding models
import openwakeword.utils as oww_utils
oww_utils.download_models()
print("✅ Base OpenWakeWord feature models ready!")

## Step 2: Upload Your Training Audio (.zip)
Upload the `alfred_training_data.zip` file generated by `tools/record_training_samples.py`.
It should contain:
- `positive/` (WAV files of you and your friend saying 'Hey Alfred' or 'Alfred')
- `negative/` (WAV files of ambient noise, typing, or random speech)

In [ ]:
from google.colab import files
import shutil

print("Please upload 'alfred_training_data.zip'...")
uploaded = files.upload()

zip_filename = list(uploaded.keys())[0]
os.makedirs("data", exist_ok=True)
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall("data")

pos_files = glob.glob("data/**/positive/*.wav", recursive=True) + glob.glob("data/positive/*.wav")
neg_files = glob.glob("data/**/negative/*.wav", recursive=True) + glob.glob("data/negative/*.wav")

print(f"\n✅ Found {len(pos_files)} positive voice clips from you & your friend!")
print(f"✅ Found {len(neg_files)} negative background/speech clips!")

## Step 3: Audio Augmentation & Background Noise Generation
To make the model robust in all room conditions, we augment your recordings with slight pitch variations, volume changes, and background noise mixing.

In [ ]:
import scipy.signal

def load_wav_16k(path):
    sr, dat = wavfile.read(path)
    if dat.ndim > 1:
        dat = dat[:, 0]
    if sr != 16000:
        dat = scipy.signal.resample(dat, int(len(dat) * 16000 / sr)).astype(np.int16)
    return dat

# Ensure plenty of negative samples if user provided fewer
if len(neg_files) < 30:
    print("Generating additional synthetic ambient noise for negative calibration...")
    os.makedirs("data/negative_synth", exist_ok=True)
    for i in range(50):
        noise = (np.random.randn(32000) * np.random.uniform(50, 400)).astype(np.int16)
        wavfile.write(f"data/negative_synth/noise_{i:02d}.wav", 16000, noise)
    neg_files.extend(glob.glob("data/negative_synth/*.wav"))

print(f"Total training pool: {len(pos_files)} positive clips, {len(neg_files)} negative clips.")

## Step 4: Extract OpenWakeWord Acoustic Feature Embeddings
We use OpenWakeWord's pretrained backbone (`embedding_model.onnx`) to convert the raw audio into 96-dimensional acoustic embedding vectors (16 frames = 1536 features).

In [ ]:
from openwakeword.model import Model

# Initialize feature extractor
oww = Model(inference_framework="onnx")
preprocessor = oww.preprocessor

X_list = []
y_list = []
step_size = 1280  # 80ms

print("Extracting features from positive clips (your voices)... ")
for p_file in tqdm(pos_files):
    audio = load_wav_16k(p_file)
    # Jitter offsets to augment real clips
    for offset in [0, 320, 640, 960]:
        sliced = audio[offset:]
        frames = []
        for i in range(0, len(sliced) - step_size, step_size):
            chunk = sliced[i:i + step_size]
            preprocessor(chunk)
            feat = preprocessor.get_features(16)  # shape (1, 16, 96)
            if feat.shape == (1, 16, 96):
                frames.append(feat)
        if frames:
            # Middle/peak frame of wake utterance
            for f in frames[len(frames)//3 : (2*len(frames))//3 + 1]:
                X_list.append(f)
                y_list.append(1.0)

print("Extracting features from negative clips (noise/chatter)... ")
for n_file in tqdm(neg_files):
    audio = load_wav_16k(n_file)
    for i in range(0, len(audio) - step_size, step_size * 2):
        chunk = audio[i:i + step_size]
        preprocessor(chunk)
        feat = preprocessor.get_features(16)
        if feat.shape == (1, 16, 96):
            X_list.append(feat)
            y_list.append(0.0)

X = np.vstack(X_list).astype(np.float32)  # shape (N, 16, 96)
y = np.array(y_list, dtype=np.float32).reshape(-1, 1)

print(f"\nDataset prepared: {len(y)} samples ({int((y==1).sum())} positive, {int((y==0).sum())} negative)")

## Step 5: Train the OpenWakeWord Neural Network
We train the standard 3-layer architecture (`1536 → 32 → 32 → 1`) with LayerNorm, ReLU, and Sigmoid.

In [ ]:
class AlfredWakeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1536, 32)
        self.norm1 = nn.LayerNorm(32)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(32, 32)
        self.norm2 = nn.LayerNorm(32)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(32, 1)
        self.sig = nn.Sigmoid()

    def forward(self, x):
        # x has shape (batch, 16, 96)
        x = x.reshape(x.shape[0], -1)
        x = self.relu1(self.norm1(self.fc1(x)))
        x = self.relu2(self.norm2(self.fc2(x)))
        return self.sig(self.fc3(x))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

dataset = TensorDataset(torch.from_numpy(X), torch.from_numpy(y))
loader = DataLoader(dataset, batch_size=32, shuffle=True)

model = AlfredWakeNet().to(device)
criterion = nn.BCELoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

epochs = 35
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for bx, by in loader:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad()
        preds = model(bx)
        loss = criterion(preds, by)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(by)
        correct += ((preds >= 0.5) == (by >= 0.5)).sum().item()
        total += len(by)

    acc = correct / total * 100.0
    if epoch % 5 == 0 or epoch == epochs:
        print(f"Epoch {epoch:02d}/{epochs:02d} - Loss: {total_loss/total:.4f} - Accuracy: {acc:.1f}%")

print("\n🎉 Neural network training complete!")

## Step 6: Export to `alfred.onnx` & Download
We export the trained model to standard ONNX with the exact input and output node signatures expected by OpenWakeWord.

In [ ]:
model.eval().to("cpu")
dummy_input = torch.randn(1, 16, 96, dtype=torch.float32)
output_onnx = "alfred.onnx"

torch.onnx.export(
    model,
    dummy_input,
    output_onnx,
    input_names=["serving_default_onnx_tf__tf_Flatten_0_eceb4355:0"],
    output_names=["PartitionedCall:0"],
    dynamic_axes={
        "serving_default_onnx_tf__tf_Flatten_0_eceb4355:0": {0: "batch"},
        "PartitionedCall:0": {0: "batch"},
    },
    opset_version=13,
)

print(f"✅ Exported '{output_onnx}' successfully!")
print(f"File size: {os.path.getsize(output_onnx) / 1024:.1f} KB")

# Trigger automatic download in browser
from google.colab import files
files.download(output_onnx)
print("📥 Your custom alfred.onnx is downloading. Drop it into Alfred's models/ folder!")